In [1]:
import torch
from torch.utils.data import DataLoader
from facenet_pytorch import MTCNN, InceptionResnetV1
from tqdm.notebook import tqdm
from datasets import load_dataset
from PIL import Image
import numpy as np
import faiss
import json
import os

# Loading Dataset

In [9]:
def load_hf_dataset(num_samples: int = 400, split: str = "train"):
    """
    Load a small sample of the HuggingFace dataset.
    """
    dataset = load_dataset("lansinuote/simple_facenet", split=split, streaming=True)
    small_sample = dataset.take(num_samples)

    images = []
    labels = []
    for sample in small_sample:
        images.append(sample["image"])
        labels.append(sample["label"])
    return images, labels

def save_to_disk(images: list[Image.Image], labels: list[str], output_dir="images"):
    """
    Save images and labels to disk.
    """
    os.makedirs(output_dir, exist_ok=True)
    for i, img in enumerate(images):
        img.save(os.path.join(output_dir, f"{i}.jpg"))
    print(f"Saved {len(images)} images to {output_dir}")
    metadata = []
    for idx, label in enumerate(labels):
        metadata.append({"image_path": f"{idx}.jpg", "label": label})
    with open(f"{output_dir}/metadata.json", "w") as f:
        json.dump(metadata, f)

def load_from_disk(images_dir ='images') -> tuple[list[Image.Image], list[dict]]:
    """
    Load all images from a directory as PIL Image objects.
    """
    image_files = sorted([f for f in os.listdir(images_dir) if f.endswith('.jpg')])
    images = []
    for filename in image_files:
        img_path = os.path.join(images_dir, filename)
        img = Image.open(img_path).convert("RGB")
        images.append(img)
    with open(f"{images_dir}/metadata.json", "r") as f:
        metadata = json.load(f)
    print(f"Loaded {len(images)} images from {images_dir}")
    return images, metadata

In [10]:
OUTPUT_DIR = "../../datasets/images/celeb_faces"

if not os.path.exists(OUTPUT_DIR):
    print("Loading dataset...")
    images, labels = load_hf_dataset(num_samples=400, split="train")
    print("Saving to disk...")
    save_to_disk(images, labels, output_dir=OUTPUT_DIR)
else:
    print("Loading images from disk...")
    images, metadata = load_from_disk(OUTPUT_DIR)

Loading images from disk...
Loaded 400 images from ../../datasets/images/celeb_faces


# Resnet Embeddings

In [11]:
# Initialize MTCNN and InceptionResnetV1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mtcnn = MTCNN(image_size=160, margin=0, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

In [12]:
def generate_embeddings(images: list[Image.Image],  mtcnn: MTCNN, resnet: InceptionResnetV1, device: str = 'cpu', batch_size: int = 32) -> np.ndarray:
    """
    Generate face embeddings for a list of images.

    Args:
        images (list): List of PIL Images or file paths to images.
        device (str): Device to run the model on ('cpu' or 'cuda').
        batch_size (int): Number of images to process in each batch.
        mtcnn (MTCNN): MTCNN model for face detection.
        resnet (InceptionResnetV1): InceptionResnetV1 model for feature extraction.

    Returns:
        list: List of face embeddings.
    """
   # Detect and align faces (list of tensors or None)
    aligned_faces = []
    for img in images:
        face = mtcnn(img)
        if face is not None:
            aligned_faces.append(face) 
    
    if len(aligned_faces) == 0:
        print("No faces detected.")
        return []

    # Stack into a tensor
    aligned_faces_tensor = torch.stack(aligned_faces)

    # Create DataLoader
    loader = DataLoader(aligned_faces_tensor, batch_size=batch_size, shuffle=False)

    embeddings = []
    resnet.eval()
    resnet.to(device)

    with torch.no_grad():
        for batch in tqdm(loader, desc="Generating embeddings"):
            batch = batch.to(device)
            emb = resnet(batch)
            embeddings.append(emb.cpu())

    embeddings_tensor = torch.cat(embeddings)
    return embeddings_tensor.numpy()

In [ ]:
embs = generate_embeddings(images, mtcnn, resnet, device=device, batch_size=32)

In [90]:
embs.shape

(399, 512)

# Vector Store

In [16]:
FAISS_DIR = "../../datasets/celeb_faces_vector_store"
FAISS_INDEX_FILE = os.path.join(FAISS_DIR, "index.faiss")
FAISS_METADATA_FILE = os.path.join(FAISS_DIR, "metadata.json")
IMAGE_DIR = "../../datasets/images/celeb_faces"
os.makedirs(FAISS_DIR, exist_ok=True)

In [17]:
class ImageFAISSVectorStore:
    def __init__(self):
        self.index = None
        self.metadata = []
        self.create_or_load_index()
    
    def create_or_load_index(self, index_file: str = FAISS_INDEX_FILE, metadata_file: str = FAISS_METADATA_FILE):
        """
        Create or load a FAISS index from the embeddings.
        """
        if os.path.exists(index_file) and os.path.exists(metadata_file):
            print(f"Loading FAISS vector store")
            self.index = faiss.read_index(index_file)
            with open(metadata_file, 'r') as f:
                self.metadata = json.load(f)
            print("Store loaded successfully.")
        else:
            print("Creating new FAISS index...")
            self.index = faiss.IndexHNSWFlat(512, 10)
        
    def add_embeddings(self, embeddings: np.ndarray, metadata: list[dict]):
        """
        Add embeddings and metadata to the FAISS index.
        
        Args:
            embeddings (np.ndarray): Array of embeddings to add.
            metadata (list[dict]): List of metadata dictionaries corresponding to the embeddings.
        """
        if self.index is None:
            raise ValueError("FAISS index is not initialized. Call create_or_load_index first.")
        
        # Add embeddings to the index
        self.index.add(embeddings)
        
        # Update metadata
        self.metadata.extend(metadata)
        
        # Save the updated index and metadata
        faiss.write_index(self.index, FAISS_INDEX_FILE)
        with open(FAISS_METADATA_FILE, 'w') as f:
            json.dump(self.metadata, f)
        
        print(f"Added {len(embeddings)} embeddings to the FAISS index.")
    
    def search(self, query_embedding: np.ndarray, k: int = 5) -> tuple[list[str], list[str], list[str]]:
        """
        Search for the k nearest neighbors of a query embedding in the FAISS index.
        
        Args:
            query_embedding (np.ndarray): The embedding to search for.
            k (int): Number of nearest neighbors to return.
        
        Returns:
            list[dict]: List of metadata dictionaries for the nearest neighbors.
        """
        if self.index is None:
            raise ValueError("FAISS index is not initialized. Call create_or_load_index first.")
        
        distances, indices = self.index.search(query_embedding.reshape(1, -1), k)
        images = []
        labels = []
        similarities = []
        for idx, dist in zip(indices[0], distances[0]):
            item = self.metadata[idx]
            images.append(f"{IMAGE_DIR}/{item['image_path']}")
            labels.append(item["label"])
            similarities.append(f"{max(0, 100 * (1 - dist / 4)):.2f}%") # Normalize L2 distance to similarity
        return images, labels, similarities

# GRADIO

In [18]:
# Load existing embeddings and metadata
vector_store = ImageFAISSVectorStore()

Loading FAISS vector store
Store loaded successfully.


In [22]:
def get_face_embedding(image: Image.Image, mtcnn: MTCNN, resnet: InceptionResnetV1, device='cpu') -> list:
    """
    Extracts the face embedding from an image using MTCNN for face detection
    and InceptionResnetV1 for feature extraction.

    Args:
        image (PIL.Image or Tensor): The input image containing a face.
        device (str): The device to run the model on ('cpu' or 'cuda').

    Returns:
        torch.Tensor: The face embedding as a 512-dimensional tensor.
    """
    # Detect and align face tensor
    face = mtcnn(image)
    if face is None:
        raise ValueError("No face detected in the image.")

    face = face.unsqueeze(0).to(device)  # add batch dimension

    with torch.no_grad():
        embedding = resnet(face).squeeze(0).cpu().numpy()

    return embedding

In [ ]:
import gradio as gr

def search_lookalikes(user_img):
    try:
        user_emb = get_face_embedding(user_img, mtcnn, resnet)
    except Exception as e:
        return f"Error: {str(e)}", [], [], []

    images, results_labels, results_similarity = vector_store.search(user_emb, k=3)
    
    results_images = []
    for img in images:
        try:
            celeb_img = Image.open(img).convert("RGB")
        except Exception:
            celeb_img = None
        
        results_images.append(celeb_img)
    
    return "Here are your top 3 celebrity lookalikes!", results_images, "---".join(results_labels), "---".join(results_similarity)

iface = gr.Interface(
    fn=search_lookalikes,
    inputs=gr.Image(type="pil", label="Upload or Capture Image"),
    outputs=[
        gr.Textbox(label="Status"),
        gr.Gallery(label="Lookalike Images", columns=[3], rows=[1], object_fit="contain", height="auto"),
        gr.Textbox(label="Labels"),
        gr.Textbox(label="Similarity")
    ],
    title="Celebrity Lookalike Finder",
    description="Take a picture and find your top 3 celebrity lookalikes!",
)

iface.launch()

In [ ]:
iface.close()  # Close the interface when done